# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

**Мои предсказания и объяснения (эталон):**

1. `{'A': 2, 'B': 1}`. `r.get(c, 0)` возвращает текущий счётчик или 0, если ключа ещё нет, и мы прибавляем 1: «A» встречается дважды, «B» один раз.
2. `[0, 1, 1]`. Список изменяемый и передаётся в функцию по ссылке, поэтому `append` внутри `f` меняет тот же самый список `a`; вызвали дважды, добавилось две единицы.
3. `2`. Множество хранит только уникальные элементы: после `.lower()` «Almaty» и «ALMATY» совпадают, остаётся `{"almaty", "astana"}`.
4. `{'ab': 2, 'abc': 3}`. Ключи в словаре уникальны: второй `"ab"` перезаписывает первый с тем же значением 2, поэтому ключей два, а не три.
5. Выведется `5`, затем `None`. Функция `g` печатает `n`, но не содержит `return`, поэтому возвращает `None`, и `x` равен `None`.

In [1]:
# Проверка предсказаний
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)


{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [4]:
# 1. Вспомогательные функции
def to_num(s):
    """'9 610' -> 9610.0 (убираем пробелы-разделители тысяч, затем float)."""
    return float(s.replace(" ", "").replace("\xa0", ""))

CITY_MAP = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда", "актобе": "Актобе", "атырау": "Атырау",
}

def clean_city(s):
    """' Алматы', 'АЛМАТЫ', 'Almaty' -> 'Алматы'."""
    return CITY_MAP[s.strip().lower()]

# проверка: все значения привелись к 6 городам, все суммы разбираются
print(sorted({clean_city(r["city"]) for r in rows}))
print(to_num("9 610"), to_num("8070.0"))

# 2. Заказов по категориям (по убыванию)
cat_count = {}
for r in rows:
    cat_count[r["category"]] = cat_count.get(r["category"], 0) + 1
for cat, n in sorted(cat_count.items(), key=lambda kv: -kv[1]):
    print(f"{cat:<12} {n}")

# 3. Выручка оплаченных и средний чек (по оплаченным) по категориям
cat_rev, cat_paid = {}, {}
for r in rows:
    if r["status"] == "оплачен":
        c = r["category"]
        cat_rev[c] = cat_rev.get(c, 0) + to_num(r["amount"])
        cat_paid[c] = cat_paid.get(c, 0) + 1

print(f"\n{'категория':<12} | {'выручка':>12} | {'ср. чек':>9}")
for c in sorted(cat_rev, key=lambda c: -cat_rev[c]):
    print(f"{c:<12} | {cat_rev[c]:>12,.0f} | {cat_rev[c] / cat_paid[c]:>9,.0f}")

# 4. Заказов по месяцам
month_count = {}
for r in rows:
    m = r["order_date"][:7]
    month_count[m] = month_count.get(m, 0) + 1
print()
for m in sorted(month_count):
    print(m, month_count[m])


['Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент']
9610.0 8070.0
Одежда       1529
Электроника  1185
Дом          1105
Красота      991
Книги        653

категория    |      выручка |   ср. чек
Электроника  |   65,270,240 |    61,287
Одежда       |   18,321,080 |    13,286
Дом          |   10,516,320 |    10,655
Красота      |    5,507,620 |     6,188
Книги        |    2,294,100 |     3,935

2026-09 2782
2026-10 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [5]:
# 5. Выручка оплаченных по месяцам и изменение окт/сен
def revenue_by_month(rs):
    out = {}
    for r in rs:
        if r["status"] == "оплачен":
            m = r["order_date"][:7]
            out[m] = out.get(m, 0) + to_num(r["amount"])
    return out

rev_month = revenue_by_month(rows)
for m in sorted(rev_month):
    print(m, f"{rev_month[m]:,.0f}")
change = (rev_month["2026-10"] / rev_month["2026-09"] - 1) * 100
print(f"Изменение октября к сентябрю: {change:.1f}%")

# 6. Доля отменённых заказов по чистым городам
total_by_city, canc_by_city = {}, {}
for r in rows:
    city = clean_city(r["city"])
    total_by_city[city] = total_by_city.get(city, 0) + 1
    if r["status"] == "отменён":
        canc_by_city[city] = canc_by_city.get(city, 0) + 1
for city in sorted(total_by_city, key=lambda c: -canc_by_city.get(c, 0) / total_by_city[c]):
    share = canc_by_city.get(city, 0) / total_by_city[city] * 100
    print(f"{city:<10} {share:.1f}%  ({canc_by_city.get(city, 0)} из {total_by_city[city]})")

# 7. Топ-5 клиентов по выручке оплаченных заказов
rev_customer = {}
for r in rows:
    if r["status"] == "оплачен":
        cid = r["customer_id"]
        rev_customer[cid] = rev_customer.get(cid, 0) + to_num(r["amount"])
top5 = sorted(rev_customer.items(), key=lambda kv: -kv[1])[:5]
for cid, rev in top5:
    print(f"{cid:<8} {rev:>14,.0f}")
top_customer = top5[0][0]

# 8. Уникальные клиенты по городам (словарь множеств)
customers_by_city = {}
for r in rows:
    city = clean_city(r["city"])
    customers_by_city.setdefault(city, set()).add(r["customer_id"])
for city in sorted(customers_by_city, key=lambda c: -len(customers_by_city[c])):
    print(f"{city:<10} {len(customers_by_city[city])}")


2026-09 57,489,850
2026-10 44,419,510
Изменение октября к сентябрю: -22.7%
Шымкент    20.4%  (164 из 804)
Актобе     10.3%  (41 из 397)
Караганда  8.9%  (48 из 538)
Астана     8.8%  (123 из 1390)
Алматы     7.9%  (172 из 2188)
Атырау     7.5%  (11 из 146)
TEST         29,999,970
C01627          257,180
C00485          215,920
C01338          192,080
C02122          191,670
Алматы     1351
Астана     981
Шымкент    655
Караганда  451
Актобе     358
Атырау     138


**Ответ на 7 (эталон).** На первом месте клиент `TEST` с выручкой 29 999 970 ₸. Это явно тестовый аккаунт: у него 3 заказа по одной и той же круглой «заглушке» 9 999 990 ₸, а у второго места (`C01627`) всего 257 180 ₸, то есть в 117 раз меньше. Его нужно исключить из аналитики. Заодно он завышает число уникальных клиентов в Алматы из задачи 8 на одного.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [6]:
# 9. Полные дубли строк через множество кортежей
seen = set()
dups = []
dedup_rows = []
for r in rows:
    key = tuple(r.values())
    if key in seen:
        dups.append(r)
    else:
        seen.add(key)
        dedup_rows.append(r)
print("Полных дублей:", len(dups))
dup_months = {}
for r in dups:
    m = r["order_date"][:7]
    dup_months[m] = dup_months.get(m, 0) + 1
print("По месяцам:", dup_months)

# 10. Выручка по месяцам без дублей и без клиента из задачи 7
clean_rows = [r for r in dedup_rows if r["customer_id"] != top_customer]
rev_clean = revenue_by_month(clean_rows)
for m in sorted(rev_clean):
    print(m, f"{rev_clean[m]:,.0f}")
change_clean = (rev_clean["2026-10"] / rev_clean["2026-09"] - 1) * 100
print(f"Было: {change:.1f}%, стало: {change_clean:.1f}%")

# 11. Клиент с наибольшим числом заказов (после удаления дублей)
orders_by_customer = {}
for r in dedup_rows:
    cid = r["customer_id"]
    orders_by_customer[cid] = orders_by_customer.get(cid, 0) + 1
best = max(orders_by_customer, key=orders_by_customer.get)
print(best, orders_by_customer[best])


Полных дублей: 180
По месяцам: {'2026-09': 180}
2026-09 35,243,410
2026-10 34,419,520
Было: -22.7%, стало: -2.3%
C01818 10


**Сообщение менеджеру (эталон, задача 12).**

Выручка оплаченных заказов в октябре упала не на 22,7%, как показал первый расчёт, а лишь на 2,3% (с 35,2 до 34,4 млн ₸). Первый расчёт был неверным из-за двух проблем в данных: в сентябре есть 180 полных дублей строк, а у тестового клиента `TEST` 3 «заказа» по 9 999 990 ₸ (около 30,0 млн ₸, из них 2 в сентябре), и всё это искусственно раздуло сентябрь. После удаления дублей и тестового аккаунта месяцы почти сопоставимы, поэтому говорить о серьёзном падении рано. Предлагаю исключать тестовые аккаунты на выгрузке и добавить в отчёт проверку на дубли.